# BTC Options Orderbook Exploratory Data Analysis

This notebook performs exploratory data analysis on BTC options orderbook data from Deribit, focusing on liquidity metrics relevant for constructing a crypto yield curve.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import ast
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

## 1. Data Loading & Parsing

In [ ]:
# Load the orderbook data
df = pd.read_csv("data/btc_option_orderbooks.csv")
print(f"Loaded {len(df):,} records")
print(f"Columns: {list(df.columns)}")
df.head(3)

In [ ]:
# Parse market name to extract components
# Format: deribit-BTC-{EXPIRY}-{STRIKE}-{C/P}-option
# Example: deribit-BTC-30JAN26-75000-P-option


def parse_market_name(market: str) -> dict:
    """Parse market name into components."""
    parts = market.split("-")
    # parts: ['deribit', 'BTC', '30JAN26', '75000', 'P', 'option']
    if len(parts) >= 6:
        expiry_str = parts[2]
        strike = int(parts[3])
        option_type = parts[4]  # 'C' or 'P'

        # Parse expiry date (e.g., '30JAN26' -> datetime)
        try:
            expiry_date = datetime.strptime(expiry_str, "%d%b%y")
        except ValueError:
            expiry_date = None

        return {
            "expiry_str": expiry_str,
            "expiry_date": expiry_date,
            "strike": strike,
            "option_type": option_type,
        }
    return {
        "expiry_str": None,
        "expiry_date": None,
        "strike": None,
        "option_type": None,
    }


# Apply parsing
parsed = df["market"].apply(parse_market_name).apply(pd.Series)
df = pd.concat([df, parsed], axis=1)

# Parse timestamp (handle mixed formats - some have microseconds, some don't)
df["time"] = pd.to_datetime(df["time"], format="mixed", utc=True)

print(f"Parsed {len(df):,} records")
print(f"\nDate range: {df['time'].min()} to {df['time'].max()}")
print(f"Unique expiries: {df['expiry_str'].nunique()}")
print(f"Unique strikes: {df['strike'].nunique()}")
print(f"Option types: {df['option_type'].value_counts().to_dict()}")

In [ ]:
# Parse the asks/bids JSON columns
# Note: ast.literal_eval is safe for parsing Python literals (unlike eval)
def parse_orderbook_side(side_str: str) -> list:
    """Parse orderbook side string to list of {price, size} dicts.

    Uses ast.literal_eval which safely evaluates Python literal structures
    (strings, numbers, tuples, lists, dicts, booleans, None).
    """
    if pd.isna(side_str) or side_str == "" or side_str == "[]":
        return []
    try:
        return ast.literal_eval(side_str)
    except (ValueError, SyntaxError):
        return []


# Parse orderbook columns (this may take a moment)
print("Parsing orderbook data...")
df["asks_parsed"] = df["asks"].apply(parse_orderbook_side)
df["bids_parsed"] = df["bids"].apply(parse_orderbook_side)
print("Done!")

In [ ]:
# Extract best bid/ask prices and sizes
def get_best_bid(bids: list) -> tuple:
    """Get best (highest) bid price and size."""
    if not bids:
        return np.nan, np.nan
    # Bids should be sorted descending by price (best first)
    best = bids[0]
    return float(best["price"]), float(best["size"])


def get_best_ask(asks: list) -> tuple:
    """Get best (lowest) ask price and size."""
    if not asks:
        return np.nan, np.nan
    # Asks should be sorted ascending by price (best first)
    best = asks[0]
    return float(best["price"]), float(best["size"])


# Extract best bid/ask
df["best_bid"], df["best_bid_size"] = zip(*df["bids_parsed"].apply(get_best_bid))
df["best_ask"], df["best_ask_size"] = zip(*df["asks_parsed"].apply(get_best_ask))

# Calculate mid price and spread
df["mid_price"] = (df["best_bid"] + df["best_ask"]) / 2
df["spread"] = df["best_ask"] - df["best_bid"]
df["spread_pct"] = df["spread"] / df["mid_price"] * 100  # Spread as % of mid

print(f"Records with valid bid: {df['best_bid'].notna().sum():,}")
print(f"Records with valid ask: {df['best_ask'].notna().sum():,}")
print(f"Records with both: {(df['best_bid'].notna() & df['best_ask'].notna()).sum():,}")

## 2. Coverage Analysis

In [ ]:
# Unique instruments overview
print("=== Instrument Coverage ===")
print(f"Total unique instruments: {df['market'].nunique():,}")
print(f"\nBy option type:")
for opt_type in ["C", "P"]:
    mask = df["option_type"] == opt_type
    print(f"  {opt_type}: {df.loc[mask, 'market'].nunique()} unique instruments")

In [ ]:
# Expiry distribution
expiry_counts = df.groupby("expiry_date")["market"].nunique().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
expiry_counts.plot(kind="bar", ax=ax, color="steelblue", alpha=0.7)
ax.set_xlabel("Expiry Date")
ax.set_ylabel("Number of Unique Instruments")
ax.set_title("Unique Instruments per Expiry Date")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print(f"\nExpiries with most instruments:")
print(expiry_counts.sort_values(ascending=False).head(10))

In [ ]:
# Strike distribution
strike_counts = df.groupby("strike")["market"].nunique().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(
    strike_counts.index, strike_counts.values, width=1000, color="darkorange", alpha=0.7
)
ax.set_xlabel("Strike Price (USD)")
ax.set_ylabel("Number of Unique Instruments")
ax.set_title("Unique Instruments per Strike Price")
ax.set_xlim(0, 250000)
plt.tight_layout()
plt.show()

print(f"\nStrike range: ${df['strike'].min():,} - ${df['strike'].max():,}")
print(f"Most common strikes:")
print(strike_counts.sort_values(ascending=False).head(10))

In [ ]:
# Heatmap: strikes vs expiries (data availability)
# Create pivot table of record counts
pivot = df.pivot_table(
    values="time", index="strike", columns="expiry_date", aggfunc="count"
).fillna(0)

# Filter to strikes with reasonable coverage (remove extremes)
strike_totals = pivot.sum(axis=1)
top_strikes = strike_totals.nlargest(50).index
pivot_filtered = pivot.loc[pivot.index.isin(top_strikes)].sort_index()

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(
    pivot_filtered, cmap="YlOrRd", ax=ax, cbar_kws={"label": "Number of Snapshots"}
)
ax.set_xlabel("Expiry Date")
ax.set_ylabel("Strike Price")
ax.set_title("Orderbook Coverage: Strikes vs Expiries (Top 50 strikes by volume)")
plt.tight_layout()
plt.show()

In [ ]:
# Time series of active instruments
df["date"] = df["time"].dt.date
daily_instruments = df.groupby("date")["market"].nunique()

fig, ax = plt.subplots(figsize=(14, 5))
daily_instruments.plot(ax=ax, color="green", linewidth=1.5)
ax.set_xlabel("Date")
ax.set_ylabel("Number of Active Instruments")
ax.set_title("Daily Count of Active Instruments")
ax.fill_between(
    daily_instruments.index, daily_instruments.values, alpha=0.3, color="green"
)
plt.tight_layout()
plt.show()

print(f"Average daily instruments: {daily_instruments.mean():.0f}")
print(f"Min: {daily_instruments.min()}, Max: {daily_instruments.max()}")

## 3. Liquidity Metrics

In [ ]:
# Filter to valid quotes only
valid_quotes = df[
    (df["best_bid"].notna()) & (df["best_ask"].notna()) & (df["spread"] >= 0)
].copy()
print(
    f"Valid two-sided quotes: {len(valid_quotes):,} ({100 * len(valid_quotes) / len(df):.1f}%)"
)

In [ ]:
# Bid-ask spread distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute spread (in BTC terms)
ax = axes[0]
spread_clipped = valid_quotes["spread"].clip(
    upper=valid_quotes["spread"].quantile(0.99)
)
ax.hist(spread_clipped, bins=50, color="purple", alpha=0.7, edgecolor="black")
ax.set_xlabel("Bid-Ask Spread (BTC)")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Absolute Bid-Ask Spread")
ax.axvline(
    valid_quotes["spread"].median(),
    color="red",
    linestyle="--",
    label=f"Median: {valid_quotes['spread'].median():.4f}",
)
ax.legend()

# Percentage spread
ax = axes[1]
spread_pct_clipped = valid_quotes["spread_pct"].clip(
    upper=valid_quotes["spread_pct"].quantile(0.99)
)
ax.hist(spread_pct_clipped, bins=50, color="teal", alpha=0.7, edgecolor="black")
ax.set_xlabel("Bid-Ask Spread (%)")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Relative Bid-Ask Spread")
ax.axvline(
    valid_quotes["spread_pct"].median(),
    color="red",
    linestyle="--",
    label=f"Median: {valid_quotes['spread_pct'].median():.1f}%",
)
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nSpread Statistics (BTC):")
print(valid_quotes["spread"].describe())
print(f"\nSpread Statistics (%):")
print(valid_quotes["spread_pct"].describe())

In [ ]:
# Calculate moneyness (requires BTC spot price assumption)
# For Jan 2026 data, BTC was around $90,000-$100,000
# We'll use mid_price as a proxy for option value to categorize moneyness

# Approximate spot price (could be refined with actual BTC price data)
BTC_SPOT_APPROX = 93000  # Approximate BTC price in Jan 2026


def categorize_moneyness(row):
    """Categorize option moneyness based on strike vs spot."""
    strike = row["strike"]
    opt_type = row["option_type"]

    # Moneyness ratio
    ratio = strike / BTC_SPOT_APPROX

    if opt_type == "C":  # Call
        if ratio < 0.95:
            return "ITM"
        elif ratio <= 1.05:
            return "ATM"
        else:
            return "OTM"
    else:  # Put
        if ratio > 1.05:
            return "ITM"
        elif ratio >= 0.95:
            return "ATM"
        else:
            return "OTM"


valid_quotes["moneyness"] = valid_quotes.apply(categorize_moneyness, axis=1)

print(f"Moneyness distribution:")
print(valid_quotes["moneyness"].value_counts())

In [ ]:
# Spread by moneyness
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot of spread by moneyness
ax = axes[0]
moneyness_order = ["ITM", "ATM", "OTM"]
valid_quotes.boxplot(
    column="spread_pct",
    by="moneyness",
    ax=ax,
    positions=[
        moneyness_order.index(m)
        for m in valid_quotes["moneyness"].unique()
        if m in moneyness_order
    ],
)
ax.set_ylim(0, valid_quotes["spread_pct"].quantile(0.95))
ax.set_xlabel("Moneyness")
ax.set_ylabel("Spread (%)")
ax.set_title("Bid-Ask Spread by Moneyness")
plt.suptitle("")

# Median spread by moneyness
ax = axes[1]
moneyness_spread = (
    valid_quotes.groupby("moneyness")["spread_pct"].median().reindex(moneyness_order)
)
colors = ["#2ecc71", "#3498db", "#e74c3c"]
bars = ax.bar(
    moneyness_order, moneyness_spread.values, color=colors, alpha=0.7, edgecolor="black"
)
ax.set_xlabel("Moneyness")
ax.set_ylabel("Median Spread (%)")
ax.set_title("Median Bid-Ask Spread by Moneyness")
for bar, val in zip(bars, moneyness_spread.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.1,
        f"{val:.1f}%",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

In [ ]:
# Calculate time to expiry
# Convert expiry_date to timezone-aware to match time column
valid_quotes["expiry_date_utc"] = pd.to_datetime(
    valid_quotes["expiry_date"]
).dt.tz_localize("UTC")
valid_quotes["tte_days"] = (
    valid_quotes["expiry_date_utc"] - valid_quotes["time"].dt.normalize()
).dt.days


# Bin time to expiry
def tte_bucket(days):
    if pd.isna(days):
        return "Unknown"
    if days <= 7:
        return "0-7d"
    elif days <= 30:
        return "7-30d"
    elif days <= 90:
        return "30-90d"
    elif days <= 180:
        return "90-180d"
    else:
        return ">180d"


valid_quotes["tte_bucket"] = valid_quotes["tte_days"].apply(tte_bucket)

# Spread by time to expiry
fig, ax = plt.subplots(figsize=(10, 5))
tte_order = ["0-7d", "7-30d", "30-90d", "90-180d", ">180d"]
tte_spread = (
    valid_quotes.groupby("tte_bucket")["spread_pct"].median().reindex(tte_order)
)
bars = ax.bar(tte_order, tte_spread.values, color="coral", alpha=0.7, edgecolor="black")
ax.set_xlabel("Time to Expiry")
ax.set_ylabel("Median Spread (%)")
ax.set_title("Median Bid-Ask Spread by Time to Expiry")
for bar, val in zip(bars, tte_spread.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.1,
        f"{val:.1f}%",
        ha="center",
        va="bottom",
    )
plt.tight_layout()
plt.show()

print(f"\nSpread stats by time to expiry:")
print(
    valid_quotes.groupby("tte_bucket")["spread_pct"]
    .agg(["count", "median", "mean", "std"])
    .reindex(tte_order)
)

In [ ]:
# Spread by option type (calls vs puts)
fig, ax = plt.subplots(figsize=(8, 5))
opt_spread = valid_quotes.groupby("option_type")["spread_pct"].median()
colors = ["#3498db", "#e74c3c"]
bars = ax.bar(
    ["Call", "Put"],
    [opt_spread.get("C", 0), opt_spread.get("P", 0)],
    color=colors,
    alpha=0.7,
    edgecolor="black",
)
ax.set_xlabel("Option Type")
ax.set_ylabel("Median Spread (%)")
ax.set_title("Median Bid-Ask Spread by Option Type")
for bar, val in zip(bars, [opt_spread.get("C", 0), opt_spread.get("P", 0)]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.1,
        f"{val:.1f}%",
        ha="center",
        va="bottom",
    )
plt.tight_layout()
plt.show()

In [ ]:
# Orderbook depth analysis
def calculate_depth(orderbook: list, mid_price: float, pct_from_mid: float) -> float:
    """Calculate total size within X% of mid price."""
    if not orderbook or pd.isna(mid_price):
        return 0.0

    threshold = mid_price * pct_from_mid / 100
    total_size = 0.0

    for level in orderbook:
        price = float(level["price"])
        size = float(level["size"])
        if abs(price - mid_price) <= threshold:
            total_size += size

    return total_size


# Calculate depth at various levels (sample for performance)
sample_size = min(50000, len(valid_quotes))
depth_sample = valid_quotes.sample(n=sample_size, random_state=42).copy()

print(f"Calculating depth metrics on {sample_size:,} samples...")

for pct in [1, 5, 10]:
    depth_sample[f"bid_depth_{pct}pct"] = depth_sample.apply(
        lambda x: calculate_depth(x["bids_parsed"], x["mid_price"], pct), axis=1
    )
    depth_sample[f"ask_depth_{pct}pct"] = depth_sample.apply(
        lambda x: calculate_depth(x["asks_parsed"], x["mid_price"], pct), axis=1
    )
    depth_sample[f"total_depth_{pct}pct"] = (
        depth_sample[f"bid_depth_{pct}pct"] + depth_sample[f"ask_depth_{pct}pct"]
    )

print("Done!")

In [ ]:
# Depth distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, pct in enumerate([1, 5, 10]):
    ax = axes[i]
    col = f"total_depth_{pct}pct"
    depth_clipped = depth_sample[col].clip(upper=depth_sample[col].quantile(0.99))
    ax.hist(depth_clipped, bins=50, color="steelblue", alpha=0.7, edgecolor="black")
    ax.set_xlabel("Depth (BTC)")
    ax.set_ylabel("Frequency")
    ax.set_title(f"Orderbook Depth within {pct}% of Mid")
    ax.axvline(
        depth_sample[col].median(),
        color="red",
        linestyle="--",
        label=f"Median: {depth_sample[col].median():.1f}",
    )
    ax.legend()

plt.tight_layout()
plt.show()

print("Depth Statistics (BTC):")
for pct in [1, 5, 10]:
    col = f"total_depth_{pct}pct"
    print(f"\n{pct}% from mid:")
    print(f"  Median: {depth_sample[col].median():.1f}")
    print(f"  Mean: {depth_sample[col].mean():.1f}")
    print(f"  75th percentile: {depth_sample[col].quantile(0.75):.1f}")

In [ ]:
# Bid/Ask depth asymmetry
depth_sample["depth_asymmetry"] = (
    depth_sample["bid_depth_5pct"] - depth_sample["ask_depth_5pct"]
) / (depth_sample["bid_depth_5pct"] + depth_sample["ask_depth_5pct"] + 1e-10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(
    depth_sample["depth_asymmetry"].clip(-1, 1),
    bins=50,
    color="purple",
    alpha=0.7,
    edgecolor="black",
)
ax.axvline(0, color="black", linestyle="-", linewidth=2)
ax.axvline(
    depth_sample["depth_asymmetry"].median(),
    color="red",
    linestyle="--",
    label=f"Median: {depth_sample['depth_asymmetry'].median():.3f}",
)
ax.set_xlabel("Depth Asymmetry (Bid - Ask) / (Bid + Ask)")
ax.set_ylabel("Frequency")
ax.set_title("Orderbook Depth Asymmetry (5% from mid)")
ax.legend()
plt.tight_layout()
plt.show()

print(
    f"Asymmetry > 0 (more bid depth): {(depth_sample['depth_asymmetry'] > 0).sum() / len(depth_sample) * 100:.1f}%"
)
print(
    f"Asymmetry < 0 (more ask depth): {(depth_sample['depth_asymmetry'] < 0).sum() / len(depth_sample) * 100:.1f}%"
)

In [ ]:
# Quote quality: % of snapshots with valid two-sided quotes
df["has_bid"] = df["best_bid"].notna()
df["has_ask"] = df["best_ask"].notna()
df["has_both"] = df["has_bid"] & df["has_ask"]

quote_quality = pd.DataFrame(
    {
        "Has Bid": [df["has_bid"].mean() * 100],
        "Has Ask": [df["has_ask"].mean() * 100],
        "Has Both": [df["has_both"].mean() * 100],
    }
).T
quote_quality.columns = ["Percentage"]

print("Quote Quality Summary:")
print(quote_quality.round(1))

# Quote quality by moneyness (using full dataset)
df["moneyness"] = df.apply(
    lambda x: categorize_moneyness(x) if pd.notna(x["strike"]) else None, axis=1
)
quality_by_moneyness = df.groupby("moneyness")["has_both"].mean() * 100
print(f"\nTwo-sided quote availability by moneyness:")
print(quality_by_moneyness.round(1))

## 4. Market Microstructure

In [ ]:
# Number of price levels per side
df["n_bid_levels"] = df["bids_parsed"].apply(len)
df["n_ask_levels"] = df["asks_parsed"].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(
    df["n_bid_levels"],
    bins=range(0, 30),
    color="green",
    alpha=0.7,
    label="Bids",
    edgecolor="black",
)
ax.hist(
    df["n_ask_levels"],
    bins=range(0, 30),
    color="red",
    alpha=0.5,
    label="Asks",
    edgecolor="black",
)
ax.set_xlabel("Number of Price Levels")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Price Levels per Side")
ax.legend()

ax = axes[1]
ax.boxplot([df["n_bid_levels"], df["n_ask_levels"]], labels=["Bids", "Asks"])
ax.set_ylabel("Number of Price Levels")
ax.set_title("Price Levels per Side")

plt.tight_layout()
plt.show()

print(f"Average bid levels: {df['n_bid_levels'].mean():.1f}")
print(f"Average ask levels: {df['n_ask_levels'].mean():.1f}")

In [ ]:
# Size concentration (% of total depth at top level)
def top_level_concentration(orderbook: list) -> float:
    """Calculate % of total size at top price level."""
    if not orderbook or len(orderbook) == 0:
        return np.nan

    total = sum(float(level["size"]) for level in orderbook)
    if total == 0:
        return np.nan

    top_size = float(orderbook[0]["size"])
    return (top_size / total) * 100


# Sample for performance
conc_sample = df.sample(n=min(50000, len(df)), random_state=42).copy()
conc_sample["bid_concentration"] = conc_sample["bids_parsed"].apply(
    top_level_concentration
)
conc_sample["ask_concentration"] = conc_sample["asks_parsed"].apply(
    top_level_concentration
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(
    conc_sample["bid_concentration"].dropna(),
    bins=50,
    color="green",
    alpha=0.6,
    label="Bid",
    edgecolor="black",
)
ax.hist(
    conc_sample["ask_concentration"].dropna(),
    bins=50,
    color="red",
    alpha=0.6,
    label="Ask",
    edgecolor="black",
)
ax.set_xlabel("Top Level Concentration (%)")
ax.set_ylabel("Frequency")
ax.set_title("Size Concentration at Top Price Level")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Median bid concentration: {conc_sample['bid_concentration'].median():.1f}%")
print(f"Median ask concentration: {conc_sample['ask_concentration'].median():.1f}%")

In [ ]:
# Time-of-day patterns in liquidity
df["hour"] = df["time"].dt.hour

# Filter to valid quotes
hourly_valid = df[df["has_both"]].copy()

# Spread by hour
hourly_spread = hourly_valid.groupby("hour")["spread_pct"].median()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.bar(
    hourly_spread.index,
    hourly_spread.values,
    color="steelblue",
    alpha=0.7,
    edgecolor="black",
)
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Median Spread (%)")
ax.set_title("Median Bid-Ask Spread by Hour of Day")
ax.set_xticks(range(0, 24))

# Active instruments by hour
hourly_instruments = df.groupby("hour")["market"].nunique()
ax = axes[1]
ax.bar(
    hourly_instruments.index,
    hourly_instruments.values,
    color="coral",
    alpha=0.7,
    edgecolor="black",
)
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Unique Instruments Quoted")
ax.set_title("Active Instruments by Hour of Day")
ax.set_xticks(range(0, 24))

plt.tight_layout()
plt.show()

## 5. Data Quality Checks

In [ ]:
# Missing/empty orderbooks
empty_bids = (df["n_bid_levels"] == 0).sum()
empty_asks = (df["n_ask_levels"] == 0).sum()
empty_both = ((df["n_bid_levels"] == 0) & (df["n_ask_levels"] == 0)).sum()

print("=== Empty Orderbook Analysis ===")
print(f"Empty bid side: {empty_bids:,} ({100 * empty_bids / len(df):.2f}%)")
print(f"Empty ask side: {empty_asks:,} ({100 * empty_asks / len(df):.2f}%)")
print(f"Both sides empty: {empty_both:,} ({100 * empty_both / len(df):.2f}%)")

In [ ]:
# Zero-size levels check
def has_zero_size(orderbook: list) -> bool:
    """Check if any level has zero size."""
    for level in orderbook:
        if float(level["size"]) == 0:
            return True
    return False


# Sample check
zero_check_sample = df.sample(n=min(50000, len(df)), random_state=42)
zero_bid = zero_check_sample["bids_parsed"].apply(has_zero_size).sum()
zero_ask = zero_check_sample["asks_parsed"].apply(has_zero_size).sum()

print("=== Zero-Size Level Check (sampled) ===")
print(
    f"Orderbooks with zero-size bids: {zero_bid:,} ({100 * zero_bid / len(zero_check_sample):.2f}%)"
)
print(
    f"Orderbooks with zero-size asks: {zero_ask:,} ({100 * zero_ask / len(zero_check_sample):.2f}%)"
)

In [ ]:
# Price inversions (crossed books)
crossed_books = df[
    (df["best_bid"].notna())
    & (df["best_ask"].notna())
    & (df["best_bid"] > df["best_ask"])
]
print("=== Crossed Book Analysis ===")
print(
    f"Crossed orderbooks: {len(crossed_books):,} ({100 * len(crossed_books) / len(df):.4f}%)"
)

if len(crossed_books) > 0:
    print(f"\nExample crossed books:")
    print(crossed_books[["market", "time", "best_bid", "best_ask"]].head())

In [ ]:
# Time series gaps
# Check hourly continuity per instrument
def check_gaps(group):
    """Check for gaps > 1 hour in time series."""
    times = group["time"].sort_values()
    if len(times) < 2:
        return 0
    diffs = times.diff().dt.total_seconds() / 3600  # Convert to hours
    gaps = (diffs > 1.5).sum()  # Count gaps > 1.5 hours
    return gaps


# Sample of instruments to check
top_instruments = df["market"].value_counts().head(20).index
gap_counts = df[df["market"].isin(top_instruments)].groupby("market").apply(check_gaps)

print("=== Time Series Gaps (top 20 instruments by volume) ===")
print(f"Total gaps > 1.5 hours: {gap_counts.sum()}")
print(f"Instruments with gaps: {(gap_counts > 0).sum()}")
if gap_counts.sum() > 0:
    print(f"\nInstruments with most gaps:")
    print(gap_counts[gap_counts > 0].sort_values(ascending=False).head())

In [ ]:
# Data quality summary
print("\n" + "=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

total_records = len(df)
valid_twosided = (df["has_both"]).sum()
uncrossed = ((df["has_both"]) & (df["best_bid"] <= df["best_ask"])).sum()

print(f"\nTotal records: {total_records:,}")
print(
    f"Two-sided quotes: {valid_twosided:,} ({100 * valid_twosided / total_records:.1f}%)"
)
print(f"Valid (uncrossed): {uncrossed:,} ({100 * uncrossed / total_records:.1f}%)")

## 6. Yield Curve Relevance Summary

In [ ]:
# Liquidity by expiry
expiry_liquidity = (
    valid_quotes.groupby("expiry_date")
    .agg({"spread_pct": "median", "market": "nunique", "time": "count"})
    .rename(
        columns={
            "spread_pct": "median_spread_pct",
            "market": "n_instruments",
            "time": "n_snapshots",
        }
    )
)

expiry_liquidity = expiry_liquidity.sort_index()
print("=== Liquidity by Expiry ===")
print(expiry_liquidity.head(15))

In [ ]:
# Recommend liquid expiries (spread < 10% and sufficient snapshots)
liquid_expiries = expiry_liquidity[
    (expiry_liquidity["median_spread_pct"] < 10)
    & (expiry_liquidity["n_snapshots"] > 1000)
]

print("\n=== Recommended Expiries for Yield Curve Construction ===")
print(f"Expiries with median spread < 10% and >1000 snapshots:")
print(liquid_expiries)

In [ ]:
# Liquidity by moneyness (for yield curve: ATM options are most important)
moneyness_liquidity = valid_quotes.groupby("moneyness").agg(
    {"spread_pct": ["median", "mean"], "market": "nunique", "time": "count"}
)
moneyness_liquidity.columns = [
    "median_spread",
    "mean_spread",
    "n_instruments",
    "n_snapshots",
]

print("\n=== Liquidity by Moneyness ===")
print(moneyness_liquidity)

In [ ]:
# Final recommendations
print("\n" + "=" * 60)
print("RECOMMENDATIONS FOR YIELD CURVE CONSTRUCTION")
print("=" * 60)

print("""
1. EXPIRY SELECTION:
   - Focus on major expiries (monthly/quarterly) which have better liquidity
   - Weekly expiries have wider spreads and less depth
   - Expiries with >1000 snapshots and <10% median spread are most reliable

2. STRIKE SELECTION:
   - ATM options (within 5% of spot) have tightest spreads
   - OTM options have wider spreads but more volume
   - ITM options are less liquid, use with caution

3. DATA FILTERING CRITERIA:
   - Require two-sided quotes (both bid and ask present)
   - Filter out crossed books (bid > ask)
   - Consider spread threshold: <10% for reliable pricing
   - Prefer snapshots with depth > 10 BTC within 5% of mid

4. TEMPORAL CONSIDERATIONS:
   - Liquidity is relatively stable across hours (24/7 market)
   - May see slightly wider spreads during Asian hours
   
5. QUOTE QUALITY:
   - Most snapshots have valid two-sided quotes
   - ATM options have highest quote availability
""")

print("\n=== KEY METRICS ===")
print(f"Dataset date range: {df['time'].min().date()} to {df['time'].max().date()}")
print(f"Total records: {len(df):,}")
print(f"Unique instruments: {df['market'].nunique():,}")
print(f"Unique expiries: {df['expiry_date'].nunique()}")
print(f"Overall median spread: {valid_quotes['spread_pct'].median():.1f}%")
print(
    f"ATM median spread: {valid_quotes[valid_quotes['moneyness'] == 'ATM']['spread_pct'].median():.1f}%"
)